In [1]:
from tensorflow.keras.preprocessing import sequence
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,Dense,LSTM
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.datasets import imdb
import tensorflow as tf
import numpy as np


In [2]:
# Set Vocabulary, Feature Dimension size and Maximum length of sentence

voc_size=10000
dim=128
max_length=500

In [3]:
# Load Data

(X_train,y_train),(X_test,y_test)=imdb.load_data(num_words=voc_size)
print(f"Training data size: {X_train.shape}, Training label size:{y_train.shape}")
print(f"Testing data size: {X_test.shape}, Testing label size: {y_test.shape}")

Training data size: (25000,), Training label size:(25000,)
Testing data size: (25000,), Testing label size: (25000,)


In [4]:
# Padding 'pre'
X_train=sequence.pad_sequences(X_train,maxlen=max_length)
X_test=sequence.pad_sequences(X_test,maxlen=max_length)
print(X_train[0])

[   0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0 

In [5]:
# Generate Model
model=Sequential()

2026-10-10 22:51:28.109748: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M1
2026-10-10 22:51:28.109954: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 8.00 GB
2026-10-10 22:51:28.109973: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 2.67 GB
2026-10-10 22:51:28.110204: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:306] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-10-10 22:51:28.110499: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:272] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


In [6]:
model.add(Embedding(voc_size,dim, input_length=max_length))
model.add(LSTM(dim))    # Default Activation funcion: 'tanh'
model.add(Dense(1,activation='sigmoid'))  # for binary classification
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, 500, 128)          1280000   
                                                                 
 lstm (LSTM)                 (None, 128)               131584    
                                                                 
 dense (Dense)               (None, 1)                 129       
                                                                 
Total params: 1411713 (5.39 MB)
Trainable params: 1411713 (5.39 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [7]:
# EarlyStopping
early_stopping=EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# Compile Model
model.compile(optimizer='adam',loss='binary_crossentropy', metrics=[
    'accuracy',
    tf.keras.metrics.Precision(name='precision'),
    tf.keras.metrics.Recall(name='recall'),
    tf.keras.metrics.AUC(name='auc')
])

In [8]:
## Train the model with early sstopping
history=model.fit(
    X_train,y_train,epochs=10,batch_size=32,
    validation_split=0.2,
    callbacks=[early_stopping]
)

Epoch 1/10


2026-10-10 22:51:48.661782: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


625/625 [==============================] - 49s 76ms/step - loss: 0.4695 - accuracy: 0.7777 - precision: 0.7770 - recall: 0.7810 - auc: 0.8586 - val_loss: 0.3710 - val_accuracy: 0.8394 - val_precision: 0.7984 - val_recall: 0.9028 - val_auc: 0.9189
Epoch 2/10
625/625 [==============================] - 46s 74ms/step - loss: 0.3082 - accuracy: 0.8709 - precision: 0.8641 - recall: 0.8812 - auc: 0.9418 - val_loss: 0.3623 - val_accuracy: 0.8528 - val_precision: 0.8175 - val_recall: 0.9036 - val_auc: 0.9249
Epoch 3/10
625/625 [==============================] - 46s 73ms/step - loss: 0.2226 - accuracy: 0.9161 - precision: 0.9145 - recall: 0.9188 - auc: 0.9689 - val_loss: 0.3255 - val_accuracy: 0.8692 - val_precision: 0.8802 - val_recall: 0.8510 - val_auc: 0.9393
Epoch 4/10
625/625 [==============================] - 46s 73ms/step - loss: 0.3163 - accuracy: 0.8619 - precision: 0.8724 - recall: 0.8490 - auc: 0.9390 - val_loss: 0.5910 - val_accuracy: 0.7056 - val_precision: 0.9324 - val_recall: 0.43

In [9]:
results = model.evaluate(X_test, y_test, verbose=0)
for metric_name, metric_value in zip(model.metrics_names, results):
    print(f"{metric_name}: {metric_value:.4f}")

loss: 0.3367
accuracy: 0.8648
precision: 0.8813
recall: 0.8432
auc: 0.9355


In [10]:
model.save('lstm_imdb.keras')